# Ticket Data Analysis

Loads and explores the ~20k work item (Incident/Request/etc.) records from a JSON file.

Set `DATA_PATH` below to point at your JSON file.

In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [ ]:
# Path to the JSON file with the work item records (adjust as needed)
DATA_PATH = Path("../data/raw/tickets.json")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    records = json.load(f)

print(f"Loaded {len(records):,} records")
records[0]

## Build the DataFrame

List-valued fields (`Affected Business or IT Services`, `Business Entity`, `Service Team(s)`, `All Comments`) are kept as-is; dates are parsed.

In [ ]:
df = pd.DataFrame(records)

date_cols = ["Created date", "Resolution date"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df["Comment count"] = df["All Comments"].apply(lambda x: len(x) if isinstance(x, list) else 0)

print(df.shape)
df.head()

## Quick overview

In [ ]:
df.info()

In [ ]:
# Missing values per column
df.isna().sum().sort_values(ascending=False)

In [ ]:
# Value counts for the key categorical fields
for col in ["Work type", "Priority", "Urgency", "Impact", "Status"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False))